# Generación de Índices de Slices para Croacia

Utiliza el CNN Selector V3 pre-entrenado para seleccionar los 5 slices más informativos de cada volumen de Croacia.

Objetivo: Generar archivo `croatia_sagittal_indices.json` con los índices pre-calculados para usar en evaluación.

## 1. Importaciones y Configuración

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/home/palodo2/tfg/acl_classifier')

# Configuración
DATA_DIR = Path('/home/palodo2/tfg/acl_classifier/data')
CROATIA_DIR = DATA_DIR / 'croatia_npy_volumes_final'
CROATIA_METADATA_CSV = CROATIA_DIR / 'metadata_final.csv'
OUTPUT_DIR = DATA_DIR / 'slice_indices_final'
CNN_SELECTOR_PATH = Path('/home/palodo2/tfg/acl_classifier/checkpoints/acl_slice_classifier_v3/best_model_f1.pth')

OUTPUT_FILE = OUTPUT_DIR / 'croatia_sagittal_indices.json'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
K = 5  # Número de slices a seleccionar

print(f"Dispositivo: {DEVICE}")
print(f"Archivo salida: {OUTPUT_FILE}")

Dispositivo: cuda:0
Archivo salida: /home/palodo2/tfg/acl_classifier/data/slice_indices_final/croatia_sagittal_indices.json


## 2. Importar Modelos desde src

In [4]:
from src.models import SimpleCNNSelector

print("Modelos importados exitosamente")

Modelos importados exitosamente


## 3. Cargar CNN Selector

In [5]:
print(f"Cargando CNN Selector desde: {CNN_SELECTOR_PATH}")

# Cargar modelo usando SimpleCNNSelector que maneja diferentes formatos de checkpoint
cnn_model = SimpleCNNSelector(checkpoint_path=str(CNN_SELECTOR_PATH), device=DEVICE)
cnn_model.eval()

print("CNN Selector cargado exitosamente")

Cargando CNN Selector desde: /home/palodo2/tfg/acl_classifier/checkpoints/acl_slice_classifier_v3/best_model_f1.pth
 CNN Selector cargado desde: /home/palodo2/tfg/acl_classifier/checkpoints/acl_slice_classifier_v3/best_model_f1.pth
  Validation F1: 0.8563
  Device: cuda:0
CNN Selector cargado exitosamente


## 4. Funciones Auxiliares

In [6]:
def load_volume_normalized(volume_path):
    """Cargar volumen normalizado a [0, 1]"""
    volume = np.load(volume_path).astype(np.float32)
    vol_min = volume.min()
    vol_max = volume.max()
    if vol_max > vol_min:
        volume = (volume - vol_min) / (vol_max - vol_min)
    return volume

def get_acl_probabilities(volume, model):
    """
    Calcular probabilidad ACL para todos los slices de un volumen.
    Input: volume - numpy array [num_slices, H, W] normalizado a [0, 1]
    Output: array de probabilidades [num_slices]
    """
    num_slices = volume.shape[0]
    acl_probs = []

    for i in range(num_slices):
        slice_2d = volume[i]  # (H, W)
        prob = model.get_acl_probability(slice_2d)
        acl_probs.append(prob)

    return np.array(acl_probs)

print("Funciones auxiliares definidas")

Funciones auxiliares definidas


## 5. Cargar Metadatos de Croacia

In [7]:
print(f"Cargando metadatos de Croacia...")
if not CROATIA_METADATA_CSV.exists():
    print(f"Error: Metadata CSV no encontrado en {CROATIA_METADATA_CSV}")
else:
    croatia_df = pd.read_csv(CROATIA_METADATA_CSV)
    print(f"Volúmenes a procesar: {len(croatia_df)}")
    print(f"Columnas: {list(croatia_df.columns)[:5]}...")

Cargando metadatos de Croacia...
Volúmenes a procesar: 917
Columnas: ['volume_id', 'new_filename', 'original_filename', 'diagnosis_3class_code', 'diagnosis_3class_name']...


## 6. Generar Índices para Croacia

In [8]:
print(f"\n{'='*80}")
print(f"GENERANDO INDICES DE SLICES PARA CROACIA (CNN SELECTOR)")
print(f"{'='*80}")
print(f"Volúmenes: {len(croatia_df)}")
print(f"Slices por volumen: K={K}")
print(f"Dispositivo: {DEVICE}")
print(f"\nProcesando...\n")

indices_dict = {}

for idx, (_, row) in enumerate(tqdm(croatia_df.iterrows(), total=len(croatia_df), desc="Generando índices")):
    volume_id = row['volume_id']
    volume_path = CROATIA_DIR / f"{volume_id:04d}.npy"

    if not volume_path.exists():
        print(f"Advertencia: Volumen no encontrado: {volume_path}")
        continue

 # Cargar volumen normalizado
    volume = load_volume_normalized(volume_path)
    num_slices = volume.shape[0]

 # Calcular probabilidad ACL para cada slice
    acl_probs = get_acl_probabilities(volume, cnn_model)

 # Seleccionar K slices con mayor probabilidad
    if num_slices <= K:
 # Si hay pocos slices, usar todos
        selected_indices = list(range(num_slices))
 # Rellenar con repetición si necesario
        while len(selected_indices) < K:
            selected_indices.append(selected_indices[-1])
        selected_indices = selected_indices[:K]
    else:
 # Seleccionar los K mejores
        selected_indices = np.argsort(acl_probs)[-K:].tolist()
        selected_indices = sorted(selected_indices)

    indices_dict[str(volume_id)] = selected_indices

print(f"\n{'='*80}")
print(f"Índices generados para {len(indices_dict)} volúmenes")


GENERANDO INDICES DE SLICES PARA CROACIA (CNN SELECTOR)
Volúmenes: 917
Slices por volumen: K=5
Dispositivo: cuda:0

Procesando...



Generando índices: 100%|██████████| 917/917 [00:11<00:00, 80.19it/s]


Índices generados para 917 volúmenes


## 7. Guardar Índices

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_FILE, 'w') as f:
    json.dump(indices_dict, f, indent=2)

print(f"Índices guardados en: {OUTPUT_FILE}")
print(f"Volúmenes procesados: {len(indices_dict)}")

# Verificar archivo
file_size = OUTPUT_FILE.stat().st_size / 1024  # KB
print(f"Tamaño del archivo: {file_size:.1f} KB")

# Mostrar ejemplo
sample_volume_id = str(list(indices_dict.keys())[0])
print(f"\nEjemplo (volumen {sample_volume_id}): índices = {indices_dict[sample_volume_id]}")

Índices guardados en: /home/palodo2/tfg/acl_classifier/data/slice_indices_final/croatia_sagittal_indices.json
Volúmenes procesados: 917
Tamaño del archivo: 44.7 KB

Ejemplo (volumen 0): índices = [0, 1, 2, 2, 2]


## 8. Verificación

In [10]:
print(f"Verificando índices generados...\n")

# Estadísticas
depths = []
for volume_id, indices in indices_dict.items():
    volume_path = CROATIA_DIR / f"{volume_id}.npy"
    if volume_path.exists():
        vol = np.load(volume_path)
        depths.append(vol.shape[0])

if depths:
    print(f"Profundidad de volúmenes:")
    print(f"  Min: {min(depths)}")
    print(f"  Max: {max(depths)}")
    print(f"  Mean: {np.mean(depths):.1f}")

print(f"\nTodos los índices tienen K={K} slices: {all(len(v) == K for v in indices_dict.values())}")
print(f"\nArchivo listo para usar en evaluación.")

Verificando índices generados...


Todos los índices tienen K=5 slices: True

Archivo listo para usar en evaluación.
